# Phase 0 spike: zero-shot Timestral on NEM prices

Backtest window: **1–31 Oct 2026**, all 5 regions. Results are produced by `uv run python -m nemcast.phase0` (see `docs/phase0_results.md`).
This notebook scores them and plots examples.

- **DA-30**: issued 12:45 NEM time on day D, scored on the next trading day
  (D+1 04:30 → D+2 04:00, 48 steps).
- **ID-5**: issued every 3 h, 5-min resolution, 12 h horizon (144 steps).
- DA-30 modes: `uni` (each region alone), `multi` (5 regions as one multivariate series),
  `uni_pd` (+ AEMO PD price as a known covariate), `uni_pd_rm` (+ PD price and PD reserve margin).
- Baselines: AEMO pre-dispatch (PD), AEMO P5MIN (first hour only), seasonal naive, persistence.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from nemcast.phase0 import score

R = Path("../data/results")
pd.set_option("display.precision", 1)

## CPU latency (2 vCPU VM)

In [ ]:
bench = pd.DataFrame(json.loads((R / "bench.json").read_text()))
bench.pivot_table(index=["case", "symmetric"], columns="model", values="seconds")

## DA-30: all models and modes, all regions

In [ ]:
da = pd.concat([pd.read_parquet(p) for p in sorted(R.glob("da30_*.parquet"))], ignore_index=True)
pts = ["aemo_pd", "naive_day", "naive_week"]
score(da, pts, ["model", "mode"])

### DA-30 by region (best Timestral config vs baselines)

In [ ]:
s = score(da, pts, ["model", "mode"])
best = s["timestral_pinball"].idxmin()
print("best:", best)
best_da = da[(da["model"] == best[0]) & (da["mode"] == best[1])]
score(best_da, pts, ["region"])

In [ ]:
def fan(df, issue, region, point_cols, ax):
    d = df[(df["issue"] == issue) & (df["region"] == region)].set_index("valid_time")
    ax.fill_between(d.index, d["q05"], d["q95"], alpha=0.15, label="P5–P95")
    ax.fill_between(d.index, d["q10"], d["q90"], alpha=0.3, label="P10–P90")
    ax.plot(d.index, d["q50"], label="Timestral P50")
    for c in point_cols:
        ax.plot(d.index, d[c], "--", label=c)
    ax.plot(d.index, d["actual"], "k", lw=1.5, label="actual")
    ax.set_title(f"{region} issued {issue:%Y-%m-%d %H:%M}")
    ax.set_ylabel("$/MWh")


issues = sorted(best_da["issue"].unique())
fig, axes = plt.subplots(5, 1, figsize=(12, 18))
for ax, r in zip(axes, ["NSW1", "QLD1", "SA1", "TAS1", "VIC1"], strict=False):
    fan(best_da, issues[len(issues) // 2], r, ["aemo_pd"], ax)
axes[0].legend(ncol=4)
fig.tight_layout()
fig.savefig("../docs/img/phase0_da30_fan.png", dpi=80)

## ID-5: by lead-time bucket

In [ ]:
ids = pd.concat([pd.read_parquet(p) for p in sorted(R.glob("id5_*.parquet"))], ignore_index=True)
ids["lead"] = pd.cut(ids["lead_min"], [0, 60, 180, 360, 720], labels=["0-1h", "1-3h", "3-6h", "6-12h"])
pts5 = ["aemo_p5min", "aemo_pd", "persistence", "naive_day"]
score(ids, pts5, ["model", "lead"])

In [ ]:
score(ids[ids["lead"] == "0-1h"], pts5, ["region"])

In [ ]:
iss = sorted(ids["issue"].unique())
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
for ax, r in zip(axes, ["NSW1", "SA1"], strict=False):
    fan(ids, iss[len(iss) // 2], r, ["aemo_p5min", "aemo_pd"], ax)
axes[0].legend(ncol=4)
fig.tight_layout()
fig.savefig("../docs/img/phase0_id5_fan.png", dpi=80)